
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Free Expanded-Universe Data Preparation
## Current S&P 500 Constituents + Yahoo Finance

This notebook prepares a several-hundred-stock dataset for the same-stock vs. cross-stock retrieval experiments without requiring WRDS/CRSP access.

---

# Important limitation

The dataset retrospectively uses a **current S&P 500 constituent snapshot**. Consequently, former constituents and delisted firms are systematically absent, the universe is not point-in-time, and the experiment is subject to survivorship bias. It should therefore be interpreted as an **expanded-universe scalability / cross-stock retrieval validation**, not as a survivorship-bias-free backtest.

---

# Data source

The notebook obtains a current S&P 500 constituent snapshot and downloads adjusted daily data from Yahoo Finance for 2000-01-01 through 2025-12-31. `auto_adjust=True` is used so split/dividend adjustments are reflected in OHLC prices.

---

# Robust download strategy

To tolerate temporary timeouts and rate limits:

1. download tickers in chunks of 25;
2. cache each ticker as Parquet;
3. skip already cached tickers on rerun;
4. retry missing tickers individually; and
5. merge the cache into one long-format Parquet file.

This permits interrupted downloads to resume without restarting from the beginning.

---

# Output

```text
/data/dataset/stock_regime_retrieval/raw/

    yahoo_sp500_current_constituents.csv
    yahoo_sp500_current_2000_2025.parquet

    yahoo_sp500_cache/
        AAPL.parquet
        MSFT.parquet
        ...
        SPY.parquet
```

`SPY` is included separately for market-context construction.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


In [ ]:

from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
import requests
from io import StringIO

warnings.filterwarnings("ignore")

DATA_ROOT = REPO_WORK_ROOT / "finance_case"
RAW_DIR = DATA_ROOT / "raw"
CACHE_DIR = RAW_DIR / "yahoo_sp500_cache"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CONSTITUENT_FILE = RAW_DIR / "yahoo_sp500_current_constituents.csv"
OUT_FILE = RAW_DIR / "yahoo_sp500_current_2000_2025.parquet"
META_FILE = RAW_DIR / "yahoo_sp500_current_2000_2025_metadata.json"

START_DATE = "2000-01-01"
END_DATE = "2026-01-01"   # yfinance end is exclusive

CHUNK_SIZE = 25
CHUNK_SLEEP_SEC = 2
INDIVIDUAL_RETRY = 3
RETRY_SLEEP_SEC = 5

print("yfinance version:", yf.__version__)
print("Output:", OUT_FILE)
print("Cache:", CACHE_DIR)


GITHUB_CONSTITUENTS_URL = (
    "https://raw.githubusercontent.com/"
    "datasets/s-and-p-500-companies/"
    "main/data/constituents.csv"
)

WIKI_URL = (
    "https://en.wikipedia.org/wiki/"
    "List_of_S%26P_500_companies"
)

HTTP_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0 Safari/537.36"
    )
}


## 1. Obtain the current S&P 500 constituent snapshot

The primary source is a public GitHub mirror of the current S&P 500 constituent table. Required columns are Symbol, Security, GICS Sector, GICS Sub-Industry, and Date added. If that source fails, the notebook falls back to the corresponding Wikipedia table using a browser-like User-Agent.

After the snapshot CSV has been saved, subsequent executions reuse the local file to keep the universe fixed. Yahoo ticker formatting is normalized by replacing `.` with `-` (e.g., `BRK.B` -> `BRK-B`).


In [ ]:

def normalize_constituent_table(current):
    """
    Normalize either the GitHub CSV or Wikipedia table to the
    columns used by the rest of this notebook.
    """
    current = current.copy()

    required_source_cols = [
        "Symbol",
        "Security",
        "GICS Sector",
        "GICS Sub-Industry",
        "Date added",
    ]

    missing = [
        c for c in required_source_cols
        if c not in current.columns
    ]

    if missing:
        raise ValueError(
            f"Constituent source is missing columns: {missing}. "
            f"Available columns: {list(current.columns)}"
        )

    constituents = current[
        required_source_cols
    ].rename(columns={
        "Symbol": "OriginalSymbol",
        "Security": "Security",
        "GICS Sector": "Sector",
        "GICS Sub-Industry": "SubIndustry",
        "Date added": "DateAdded",
    })

    constituents["Ticker"] = (
        constituents["OriginalSymbol"]
        .astype(str)
        .str.replace(".", "-", regex=False)
    )

    constituents["SnapshotDate"] = (
        pd.Timestamp.today().normalize()
    )

    return constituents


if CONSTITUENT_FILE.exists():
    constituents = pd.read_csv(
        CONSTITUENT_FILE
    )

    print(
        "Loaded existing constituent snapshot:",
        CONSTITUENT_FILE,
    )

else:
    constituents = None

    # ---------------------------------------------------------
    # Primary source: public GitHub CSV.
    # This avoids Wikipedia's occasional HTTP 403 for scripts.
    # ---------------------------------------------------------
    try:
        print(
            "Downloading current S&P 500 constituents "
            "from GitHub..."
        )

        response = requests.get(
            GITHUB_CONSTITUENTS_URL,
            headers=HTTP_HEADERS,
            timeout=30,
        )

        response.raise_for_status()

        current = pd.read_csv(
            StringIO(response.text)
        )

        constituents = (
            normalize_constituent_table(
                current
            )
        )

        print(
            "GitHub constituent download succeeded."
        )

    except Exception as github_error:
        print(
            "GitHub source failed:",
            repr(github_error),
        )

        # -----------------------------------------------------
        # Fallback: Wikipedia with a browser-like User-Agent.
        # -----------------------------------------------------
        print(
            "Trying Wikipedia fallback with User-Agent..."
        )

        response = requests.get(
            WIKI_URL,
            headers=HTTP_HEADERS,
            timeout=30,
        )

        response.raise_for_status()

        tables = pd.read_html(
            StringIO(response.text)
        )

        current = tables[0].copy()

        constituents = (
            normalize_constituent_table(
                current
            )
        )

        print(
            "Wikipedia fallback succeeded."
        )

    constituents.to_csv(
        CONSTITUENT_FILE,
        index=False,
    )

    print(
        "Saved constituent snapshot:",
        CONSTITUENT_FILE,
    )

print(
    "Constituents:",
    len(constituents),
)

display(
    constituents.head()
)

display(
    constituents[
        "Sector"
    ].value_counts()
)


## 2. Download helper

Cache adjusted Close and Volume for each ticker. `SPY` is stored using the same format.


In [ ]:

def safe_cache_name(ticker):
    return ticker.replace("/", "_").replace("\\", "_")

def cache_path(ticker):
    return CACHE_DIR / f"{safe_cache_name(ticker)}.parquet"

def normalize_single_ticker_df(df, ticker):
    """
    Convert one ticker's yfinance output to:
    Date, Ticker, Close, Volume
    """
    if df is None or len(df) == 0:
        return None

    x = df.copy()

    if isinstance(x.columns, pd.MultiIndex):
        # For a single ticker, yfinance may still return MultiIndex.
        if ticker in x.columns.get_level_values(0):
            x = x[ticker].copy()
        elif ticker in x.columns.get_level_values(-1):
            x = x.xs(ticker, axis=1, level=-1).copy()

    x = x.reset_index()

    if "Date" not in x.columns:
        # Sometimes index name can vary.
        date_candidates = [
            c for c in x.columns
            if str(c).lower() in {"date", "datetime"}
        ]
        if not date_candidates:
            return None
        x = x.rename(columns={date_candidates[0]: "Date"})

    required = ["Date", "Close"]

    if not all(c in x.columns for c in required):
        return None

    keep = ["Date", "Close"]

    if "Volume" in x.columns:
        keep.append("Volume")

    x = x[keep].copy()

    x["Date"] = pd.to_datetime(x["Date"]).dt.tz_localize(None)
    x["Close"] = pd.to_numeric(x["Close"], errors="coerce")

    if "Volume" in x.columns:
        x["Volume"] = pd.to_numeric(x["Volume"], errors="coerce")
    else:
        x["Volume"] = np.nan

    x["Ticker"] = ticker

    x = (
        x.dropna(subset=["Date", "Close"])
        .sort_values("Date")
        .drop_duplicates("Date", keep="last")
        .reset_index(drop=True)
    )

    return x[
        ["Date", "Ticker", "Close", "Volume"]
    ]

def extract_from_batch(raw, ticker):
    if raw is None or len(raw) == 0:
        return None

    try:
        if isinstance(raw.columns, pd.MultiIndex):
            # group_by='ticker' normally puts ticker at level 0.
            if ticker in raw.columns.get_level_values(0):
                return normalize_single_ticker_df(
                    raw[ticker],
                    ticker,
                )

            # Defensive fallback.
            if ticker in raw.columns.get_level_values(-1):
                sub = raw.xs(
                    ticker,
                    axis=1,
                    level=-1,
                )
                return normalize_single_ticker_df(
                    sub,
                    ticker,
                )

        # Single ticker fallback.
        return normalize_single_ticker_df(
            raw,
            ticker,
        )

    except Exception:
        return None


## 3. Chunked download with checkpoint cache

Tickers with an existing cache file are skipped. The first pass downloads batches of 25; any missing tickers are retried individually afterward.


In [ ]:

tickers = sorted(
    constituents["Ticker"].dropna().astype(str).unique().tolist()
)

# SPY is market context, not an equity-universe query target.
download_tickers = sorted(set(tickers + ["SPY"]))

cached = [
    t for t in download_tickers
    if cache_path(t).exists()
]

missing = [
    t for t in download_tickers
    if not cache_path(t).exists()
]

print("Total tickers including SPY:", len(download_tickers))
print("Already cached:", len(cached))
print("Need download:", len(missing))


In [ ]:

failed_batch = []

for i in range(0, len(missing), CHUNK_SIZE):
    chunk = missing[i:i + CHUNK_SIZE]

    print(
        f"\nChunk {i // CHUNK_SIZE + 1}: "
        f"{len(chunk)} tickers"
    )

    try:
        raw = yf.download(
            tickers=chunk,
            start=START_DATE,
            end=END_DATE,
            interval="1d",
            auto_adjust=True,
            actions=False,
            threads=8,
            group_by="ticker",
            progress=False,
            timeout=30,
        )

    except Exception as e:
        print("Batch failed:", repr(e))
        failed_batch.extend(chunk)
        time.sleep(CHUNK_SLEEP_SEC)
        continue

    for ticker in chunk:
        df = extract_from_batch(raw, ticker)

        if df is None or len(df) == 0:
            failed_batch.append(ticker)
            continue

        df.to_parquet(
            cache_path(ticker),
            index=False,
        )

    print(
        "Cached so far:",
        sum(cache_path(t).exists() for t in download_tickers),
        "/",
        len(download_tickers),
    )

    time.sleep(CHUNK_SLEEP_SEC + random.random())

failed_batch = sorted(set(failed_batch))

print("\nBatch-missing tickers:", len(failed_batch))
print(failed_batch[:50])


## 4. Retry failed or missing tickers individually

Retry temporarily unavailable Yahoo Finance tickers with a slower per-ticker schedule.


In [ ]:

still_missing = [
    t for t in download_tickers
    if not cache_path(t).exists()
]

final_failed = []

for ticker in still_missing:
    ok = False

    for attempt in range(1, INDIVIDUAL_RETRY + 1):
        print(
            f"{ticker}: attempt {attempt}/{INDIVIDUAL_RETRY}"
        )

        try:
            raw = yf.download(
                tickers=ticker,
                start=START_DATE,
                end=END_DATE,
                interval="1d",
                auto_adjust=True,
                actions=False,
                threads=False,
                progress=False,
                timeout=30,
            )

            df = normalize_single_ticker_df(
                raw,
                ticker,
            )

            if df is not None and len(df) > 0:
                df.to_parquet(
                    cache_path(ticker),
                    index=False,
                )
                ok = True
                break

        except Exception as e:
            print("  error:", repr(e))

        time.sleep(
            RETRY_SLEEP_SEC * attempt
        )

    if not ok:
        final_failed.append(ticker)

print("\nFinal failures:", len(final_failed))
print(final_failed)


## 5. Merge cached ticker files

Attach the fixed constituent metadata to the cached price data. `SPY` is marked with `IsMarket=True`.


In [ ]:

meta_cols = [
    "Ticker",
    "OriginalSymbol",
    "Security",
    "Sector",
    "SubIndustry",
    "DateAdded",
    "SnapshotDate",
]

meta = constituents[
    [c for c in meta_cols if c in constituents.columns]
].copy()

frames = []

for ticker in download_tickers:
    p = cache_path(ticker)

    if not p.exists():
        continue

    df = pd.read_parquet(p)

    if ticker == "SPY":
        df["OriginalSymbol"] = "SPY"
        df["Security"] = "SPDR S&P 500 ETF Trust"
        df["Sector"] = "Market"
        df["SubIndustry"] = "Market ETF"
        df["DateAdded"] = np.nan
        df["SnapshotDate"] = constituents["SnapshotDate"].iloc[0]
        df["IsMarket"] = True

    else:
        row = meta[
            meta["Ticker"] == ticker
        ].iloc[0]

        for c in [
            "OriginalSymbol",
            "Security",
            "Sector",
            "SubIndustry",
            "DateAdded",
            "SnapshotDate",
        ]:
            df[c] = row[c] if c in row.index else np.nan

        df["IsMarket"] = False

    frames.append(df)

data = pd.concat(
    frames,
    ignore_index=True,
)

data["Date"] = pd.to_datetime(
    data["Date"]
).dt.tz_localize(None)

data = (
    data.sort_values(["Ticker", "Date"])
    .drop_duplicates(["Ticker", "Date"], keep="last")
    .reset_index(drop=True)
)

data.to_parquet(
    OUT_FILE,
    index=False,
)

print("Rows:", len(data))
print("Stocks:", data.loc[~data["IsMarket"], "Ticker"].nunique())
print("Date:", data["Date"].min(), "->", data["Date"].max())

display(data.head())


## 6. Coverage diagnostics

Recently listed stocks do not have observations back to 2000. We retain this natural history-length variation and use it in the same-stock vs. cross-stock coverage analysis rather than dropping the affected stocks.


In [ ]:

coverage = (
    data.loc[~data["IsMarket"]]
    .groupby(
        ["Ticker", "Security", "Sector"],
        dropna=False,
    )
    .agg(
        FirstDate=("Date", "min"),
        LastDate=("Date", "max"),
        NDays=("Date", "size"),
    )
    .reset_index()
    .sort_values("FirstDate")
)

display(coverage.head(20))
display(coverage.tail(20))

print("Stocks with data before 2005:",
      int((coverage["FirstDate"] < "2005-01-01").sum()))

print("Stocks with data before 2010:",
      int((coverage["FirstDate"] < "2010-01-01").sum()))

print("Stocks first observed 2020+:",
      int((coverage["FirstDate"] >= "2020-01-01").sum()))

coverage.to_csv(
    RAW_DIR / "yahoo_sp500_current_history_coverage.csv",
    index=False,
)


## 7. Save reproducibility metadata

In [ ]:

metadata = {
    "dataset": "Yahoo Finance current S&P 500 expanded-universe pilot",
    "universe_definition": "Current S&P 500 constituent snapshot",
    "survivorship_bias": True,
    "point_in_time_universe": False,
    "start": START_DATE,
    "end_exclusive": END_DATE,
    "auto_adjust": True,
    "stock_count_downloaded": int(
        data.loc[~data["IsMarket"], "Ticker"].nunique()
    ),
    "failed_tickers": final_failed,
    "constituent_snapshot_file": str(CONSTITUENT_FILE),
    "output_file": str(OUT_FILE),
}

with open(META_FILE, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


# Next notebook

After the following file has been created,

```text
_work/finance_case/raw/yahoo_sp500_current_2000_2025.parquet
```

run:

```text
07_finance_crossstock_ablation.ipynb
```

The core question is whether future-compatible analogs drawn from the full stock universe add value beyond analogs restricted to the same stock.
